# Lab 07 — 01 Source Profile



In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


In [ ]:
from pyspark.sql import functions as F
landing=spark.table(f'{catalog}.{schema}.business_license_landing'); feed=spark.table(f'{catalog}.{schema}.business_license_snapshot_feed')
display(landing.groupBy('application_type').count().orderBy(F.desc('count')))
profile=feed.groupBy('snapshot_version').agg(F.count('*').alias('rows'),F.countDistinct('license_number').alias('distinct_keys')).withColumn('duplicate_keys',F.col('rows')-F.col('distinct_keys'))
display(profile); assert profile.filter('duplicate_keys>0').count()==0
print('license_number is unique inside generated snapshots: PASS')
